In [1]:
import pandas as pd

/Users/chenxuxie/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [3]:
file_path = "../data/lab4_processed_tweets.csv"
df = pd.read_csv(file_path)

engagement_summary = (
    df
    .groupby("sentiment")
    .agg(
        tweet_count=("id", "count"),
        avg_likes=("like_count", "mean"),
        median_likes=("like_count", "median"),
        avg_retweets=("retweet_count", "mean"),
        median_retweets=("retweet_count", "median")
    )
    .reset_index()
)

print(engagement_summary)

print("\nLikes distribution:")
print(df["like_count"].describe())

print("\nTweets with 0 likes:")
print((df["like_count"] == 0).sum())

print("\nTweets with > 0 likes:")
print((df["like_count"] > 0).sum())

print("\nTop 10 tweets by likes:")
print(
    df[[
        "text",
        "sentiment",
        "sentiment_score",
        "like_count",
        "retweet_count"
    ]]
    .sort_values("like_count", ascending=False)
    .head(10)
    .to_string(index=False)
)

  sentiment  tweet_count  avg_likes  median_likes  avg_retweets  \
0  Negative          177   2.564972           0.0      0.288136   
1   Neutral         1088  10.102941           0.0      2.274816   
2  Positive          735   9.229932           0.0      1.668027   

   median_retweets  
0              0.0  
1              0.0  
2              0.0  

Likes distribution:
count    2000.000000
mean        9.115000
std        56.574103
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max      1054.000000
Name: like_count, dtype: float64

Tweets with 0 likes:
1264

Tweets with > 0 likes:
736

Top 10 tweets by likes:
                                                                                                                                                                                                                                                                                                                   text sentiment  sentiment_score  like_

In [8]:
col = ["retweet_count", "reply_count", "like_count", "quote_count", "bookmark_count", "impression_count"]
for c in col:
    print(f"\n{c} distribution:")
    df["engagement_level"] = pd.cut(
        df[c],
        bins=[-1, 0, 5, 20, 100, float("inf")],
        labels=["0", "1–5", "6–20", "21–100", "100+"]
    )

    engagement_table = pd.crosstab(
        df["sentiment"],
        df["engagement_level"],
        normalize="index"
    ) * 100

    print(engagement_table.round(2))


retweet_count distribution:
engagement_level      0    1–5  6–20  21–100  100+
sentiment                                         
Negative          92.09   6.78  0.56    0.56  0.00
Neutral           82.72  12.50  2.67    1.65  0.46
Positive          84.90  11.56  1.77    1.36  0.41

reply_count distribution:
engagement_level      0    1–5  6–20  21–100  100+
sentiment                                         
Negative          81.36  17.51  1.13    0.00  0.00
Neutral           83.18  13.33  1.65    1.75  0.09
Positive          77.01  18.23  2.72    1.36  0.68

like_count distribution:
engagement_level      0    1–5  6–20  21–100  100+
sentiment                                         
Negative          62.71  27.12  7.34    2.26  0.56
Neutral           64.61  23.07  5.88    4.41  2.02
Positive          61.22  27.62  4.76    4.22  2.18

quote_count distribution:
engagement_level      0   1–5  6–20  21–100
sentiment                                  
Negative          96.61  3.39  0.00   

In [5]:
from scipy.stats import spearmanr

corr, p = spearmanr(
    df["sentiment_score"],
    df["like_count"]
)

print("Spearman correlation:", corr)
print("p-value:", p)
for col in [
    "like_count",
    "retweet_count",
    "reply_count",
    "quote_count",
    "bookmark_count",
    "impression_count"
]:
    corr, p = spearmanr(
        df["sentiment_score"],
        df[col]
    )
    print(f"{col}: rho={corr:.4f}, p={p:.4g}")

Spearman correlation: 0.026407470930314703
p-value: 0.23782306393520236
like_count: rho=0.0264, p=0.2378
retweet_count: rho=0.0243, p=0.2771
reply_count: rho=0.0374, p=0.09407
quote_count: rho=0.0250, p=0.264
bookmark_count: rho=0.0259, p=0.2478
impression_count: rho=-0.0580, p=0.009514


In [6]:
engagement_score = (
    df
    .groupby("engagement_level")
    .agg(
        count=("id", "count"),
        mean_sentiment=("sentiment_score", "mean"),
        median_sentiment=("sentiment_score", "median"),
        mean_impressions=("impression_count", "mean"),
        median_impressions=("impression_count", "median")
    )
    .reset_index()
)

print(engagement_score.round(3))

  engagement_level  count  mean_sentiment  median_sentiment  mean_impressions  \
0          0 likes   1264           0.290             0.252            21.680   
1              1–5    502           0.320             0.319           142.759   
2             6–20    112           0.219             0.245          1002.741   
3           21–100     83           0.323             0.320          3194.217   
4             100+     39           0.364             0.270        108382.795   

   median_impressions  
0                 9.0  
1                43.5  
2               544.0  
3              1843.0  
4             12583.0  


/var/folders/qd/wghp74y13071cc79s__3c0xm0000gn/T/ipykernel_71201/3912048963.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("engagement_level")


In [9]:
# Total engagement
df["eng"] = (
    df["like_count"]
    + df["retweet_count"]
    + df["reply_count"]
    + df["quote_count"]
    + df["bookmark_count"]
)
def calculate_ratio(value, impressions):
    if impressions == 0:
        return 0
    return value / impressions
df["like_ratio"] = df.apply(
    lambda row: calculate_ratio(
        row["like_count"],
        row["impression_count"]
    ),
    axis=1
)

df["retweet_ratio"] = df.apply(
    lambda row: calculate_ratio(
        row["retweet_count"],
        row["impression_count"]
    ),
    axis=1
)

df["reply_ratio"] = df.apply(
    lambda row: calculate_ratio(
        row["reply_count"],
        row["impression_count"]
    ),
    axis=1
)

df["quote_ratio"] = df.apply(
    lambda row: calculate_ratio(
        row["quote_count"],
        row["impression_count"]
    ),
    axis=1
)

df["bookmark_ratio"] = df.apply(
    lambda row: calculate_ratio(
        row["bookmark_count"],
        row["impression_count"]
    ),
    axis=1
)

df["eng_ratio"] = df.apply(
    lambda row: calculate_ratio(
        row["eng"],
        row["impression_count"]
    ),
    axis=1
)

In [10]:
print(
    df[
        [
            "impression_count",
            "like_count",
            "retweet_count",
            "reply_count",
            "quote_count",
            "bookmark_count",
            "eng",
            "like_ratio",
            "retweet_ratio",
            "reply_ratio",
            "quote_ratio",
            "bookmark_ratio",
            "eng_ratio"
        ]
    ].head(10)
)
print("impressions = 0:")
print(
    df[df["impression_count"] == 0][
        [
            "impression_count",
            "eng",
            "like_ratio",
            "retweet_ratio",
            "eng_ratio"
        ]
    ].head(10)
)

   impression_count  like_count  retweet_count  reply_count  quote_count  \
0                 1           0              0            0            0   
1                 3           0              0            0            0   
2                 0           0              0            0            0   
3                70           0              0            0            0   
4                 4           0              0            0            0   
5                 3           1              1            0            0   
6                 1           0              0            0            0   
7                11           0              0            0            0   
8                 2           0              0            1            0   
9                 1           0              0            0            0   

   bookmark_count  eng  like_ratio  retweet_ratio  reply_ratio  quote_ratio  \
0               0    0    0.000000       0.000000          0.0          0.0   
1    

In [11]:
def is_ignored(df, criterion, k):
    if criterion == "eng_zero":
        return df["eng"] == 0

    elif criterion == "likes_zero":
        return df["like_count"] == 0

    elif criterion == "retweets_zero":
        return df["retweet_count"] == 0

    elif criterion == "replies_zero":
        return df["reply_count"] == 0

    elif criterion == "quotes_zero":
        return df["quote_count"] == 0

    elif criterion == "bookmarks_zero":
        return df["bookmark_count"] == 0

    elif criterion == "like_ratio_low":
        return df["like_ratio"] < k

    elif criterion == "retweet_ratio_low":
        return df["retweet_ratio"] < k

    elif criterion == "reply_ratio_low":
        return df["reply_ratio"] < k

    elif criterion == "quote_ratio_low":
        return df["quote_ratio"] < k

    elif criterion == "bookmark_ratio_low":
        return df["bookmark_ratio"] < k

    elif criterion == "eng_ratio_low":
        return df["eng_ratio"] < 5 * k

    elif criterion == "impressions_zero":
        return df["impression_count"] == 0

    else:
        raise ValueError("Unknown criterion")
def get_metric(df, metric):
    if metric == "eng":
        return df["eng"]

    elif metric == "likes":
        return df["like_count"]

    elif metric == "retweets":
        return df["retweet_count"]

    elif metric == "replies":
        return df["reply_count"]

    elif metric == "quotes":
        return df["quote_count"]

    elif metric == "bookmarks":
        return df["bookmark_count"]

    elif metric == "eng_ratio":
        return df["eng_ratio"]

    elif metric == "like_ratio":
        return df["like_ratio"]

    elif metric == "retweet_ratio":
        return df["retweet_ratio"]

    elif metric == "reply_ratio":
        return df["reply_ratio"]

    elif metric == "quote_ratio":
        return df["quote_ratio"]

    elif metric == "bookmark_ratio":
        return df["bookmark_ratio"]

    else:
        raise ValueError("Unknown metric")

In [ ]:
k = 0.01

ignored = is_ignored(
    df,
    "eng_zero",
    k
)

print("Ignored:", ignored.sum())
print("Not ignored:", (~ignored).sum())
